# 01_EDA.ipynb: Semisupervised Sensor EDA & Alarm Horizon Oracle Diagnostic (Version 2)

Tuân thủ nghiêm ngặt **AGENTS.md** và **PROCESS.md**:
- Tái cấu trúc hoàn toàn mới (Rebuild Stage 1) theo Decision Charter đã thống nhất.
- Import trực tiếp các API chuẩn hóa từ `02_Preprocessing.ipynb` (không trùng lặp mã nguồn).
- Phân tích khám phá dữ liệu cảm biến CHỈ TRÊN TẬP PHÁT TRIỂN (`ts < 2008-10-02 20:54:00`, 1253 quan sát, 87 Fail).
- Tích hợp công cụ chẩn đoán giới hạn hình học cửa sổ cảnh báo (**Hindsight Oracle Diagnostic via MILP**) trên 3 chu kỳ phát triển cố định [2008-08-21 18:00, 2008-10-02 18:00) (879 quan sát, 35 Fail).
- Không tự động gán nhãn hay áp dụng giải pháp Oracle làm mô hình dự báo vận hành.

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import milp, Bounds, LinearConstraint

# 1. Bootstrap project environment and import APIs from 02_Preprocessing
def find_project_root(start_path=None):
    p = Path(start_path).resolve() if start_path else Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "AGENTS.md").exists() and (parent / "data" / "secom").exists():
            return parent
    raise RuntimeError(f"Could not locate project root from {p}")

PROJECT_ROOT = find_project_root()
VERSION_DIR = PROJECT_ROOT / "Version 2"

if str(VERSION_DIR) not in sys.path:
    sys.path.insert(0, str(VERSION_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Execute 02_Preprocessing notebook definitions in current namespace
p02_path = VERSION_DIR / "02_Preprocessing.ipynb"
assert p02_path.exists(), f"Cannot find {p02_path}"

import json
with open(p02_path, "r", encoding="utf-8") as f:
    nb02_content = json.load(f)

for cell in nb02_content["cells"]:
    if cell["cell_type"] == "code":
        code_str = "".join(cell["source"])
        # Skip automated test execution cell when importing
        if "run_synthetic_tests()" in code_str:
            continue
        exec(code_str, globals())

print("[BOOTSTRAP] Successfully imported 02_Preprocessing APIs into 01_EDA environment.")


[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] Successfully imported 02_Preprocessing APIs into 01_EDA environment.


In [2]:
# 2. Load Raw Data & Strict Chronological Partitioning
df_raw = load_raw()

DEV_CUTOFF = pd.Timestamp("2008-10-02 20:54:00")
dev_mask = df_raw["timestamp"] < DEV_CUTOFF
dev_df = df_raw.loc[dev_mask].copy().reset_index(drop=True)
test_df = df_raw.loc[~dev_mask].copy().reset_index(drop=True)

print(f"Dataset Partitioning Summary:")
print(f"  Total Observations: {len(df_raw)} (Fails: {df_raw['fail'].sum()})")
print(f"  Development Set:    {len(dev_df)} (Fails: {dev_df['fail'].sum()})")
print(f"  Holdout Set (B):    {len(test_df)} (Fails: {test_df['fail'].sum()}) [PREVIOUSLY EXPOSED / RETROSPECTIVE ONLY]")

assert len(dev_df) == 1253 and dev_df['fail'].sum() == 87, "Development split mismatch!"
assert len(test_df) == 314 and test_df['fail'].sum() == 17, "Holdout split mismatch!"


Dataset Partitioning Summary:
  Total Observations: 1567 (Fails: 104)
  Development Set:    1253 (Fails: 87)
  Holdout Set (B):    314 (Fails: 17) [PREVIOUSLY EXPOSED / RETROSPECTIVE ONLY]


### Bối cảnh Đặc trưng & Điểm bất thường của Tập dữ liệu (Anonymous 590 Features)

1. **Tính chất ẩn danh:** Tập dữ liệu UCI SECOM bao gồm 590 cột cảm biến liên tục được ẩn danh hoàn toàn (`sensor_0` đến `sensor_589`). Không có thông tin về loại cảm biến (nhiệt độ, áp suất, lưu lượng khí, quang học...), không có đơn vị đo và không có mã định danh thiết bị/buồng bán dẫn (tool/chamber ID).
2. **Sự không nhất quán về số lượng đặc trưng:** Tài liệu giới thiệu ban đầu của UCI SECOM ghi nhận 591 đặc trưng, nhưng tệp dữ liệu thực tế `secom.data` chỉ có chính xác 590 cột. Sự sai lệch 1 đặc trưng này chưa bao giờ được làm rõ trong các tài liệu công bố chính thức.
3. **Tương quan đồng hàng vs. Khả năng dự báo trước:** Các phân tích tương quan giữa phép đo cảm biến và nhãn kiểm tra chất lượng (QC) trên cùng một hàng (same-row association) chỉ phản ánh mối liên hệ in-sample. Nó hoàn toàn **KHÔNG PHẢI là bằng chứng chứng minh khả năng dự báo trước sự cố (forecast proof)** trong vận hành tương lai, bởi vì kết quả QC thực tế cần độ trễ đo kiểm $D$ mới sẵn sàng.

In [3]:
# 3. Missingness & Variance Audit strictly on Development Data
sensor_cols = [c for c in dev_df.columns if c.startswith("sensor_")]
dev_sensors = dev_df[sensor_cols]

total_cells = dev_sensors.size
missing_cells = int(dev_sensors.isna().sum().sum())
missing_rate = missing_cells / total_cells

# Constant columns on development
constant_cols = dev_sensors.columns[dev_sensors.nunique(dropna=True) <= 1].tolist()

# Missingness distribution by feature
col_missing = dev_sensors.isna().mean()
missing_gt_50 = col_missing[col_missing > 0.5].index.tolist()

print(f"Development Features Audit:")
print(f"  Total sensor cells: {total_cells}")
print(f"  Missing cells:      {missing_cells} ({missing_rate * 100:.2f}%)")
print(f"  Constant features:  {len(constant_cols)} columns (<=1 unique non-missing value)")
print(f"  Features >50% miss: {len(missing_gt_50)} columns")

assert missing_cells == 34831 and total_cells == 739270, f"Expected 34831/739270 missing, got {missing_cells}/{total_cells}"
assert len(constant_cols) == 122, f"Expected 122 constant columns, got {len(constant_cols)}"
assert len(missing_gt_50) == 24, f"Expected 24 columns with >50% missing, got {len(missing_gt_50)}"


Development Features Audit:
  Total sensor cells: 739270
  Missing cells:      34831 (4.71%)
  Constant features:  122 columns (<=1 unique non-missing value)
  Features >50% miss: 24 columns


In [4]:
# 4. Exploratory Univariate Association on Development (Exploratory AUC Top-5)
from sklearn.metrics import roc_auc_score

auc_results = []
for col in sensor_cols:
    if col in constant_cols:
        continue
    valid = dev_df[[col, "fail"]].dropna()
    n_pass = int((valid["fail"] == 0).sum())
    n_fail = int((valid["fail"] == 1).sum())
    # Audit eligibility rule: non-constant, n_pass >= 20, n_fail >= 10
    if n_pass >= 20 and n_fail >= 10:
        try:
            score = roc_auc_score(valid["fail"], valid[col])
            auc_val = max(score, 1.0 - score)
            auc_results.append((col, auc_val, score, n_pass, n_fail, len(valid)))
        except Exception:
            pass

df_auc = pd.DataFrame(auc_results, columns=["sensor", "abs_auc", "raw_auc", "n_pass", "n_fail", "n_valid"]).sort_values("abs_auc", ascending=False)
print("Top-5 Exploratory Univariate Same-Row AUC on Development (Eligible: non-const, nPass>=20, nFail>=10):")
print(df_auc.head(5).to_string(index=False))
print("\n[DISCLAIMER] Các hệ số AUC trên chỉ là thống kê tương quan cùng hàng (in-sample univariate association) trên tập Development.")
print("Tuyệt đối không sử dụng danh sách này làm tập đặc trưng cố định cho bài toán chuỗi thời gian hay suy diễn khả năng dự báo trước.")


Top-5 Exploratory Univariate Same-Row AUC on Development (Eligible: non-const, nPass>=20, nFail>=10):
    sensor  abs_auc  raw_auc  n_pass  n_fail  n_valid
 sensor_59 0.727530 0.727530    1159      87     1246
sensor_247 0.676024 0.676024     762      69      831
sensor_519 0.675159 0.675159     762      69      831
sensor_103 0.645638 0.645638    1164      87     1251
sensor_385 0.634058 0.634058     762      69      831

[DISCLAIMER] Các hệ số AUC trên chỉ là thống kê tương quan cùng hàng (in-sample univariate association) trên tập Development.
Tuyệt đối không sử dụng danh sách này làm tập đặc trưng cố định cho bài toán chuỗi thời gian hay suy diễn khả năng dự báo trước.


## 5. Hindsight Oracle Diagnostic via Mixed-Integer Linear Programming (MILP)

> [!WARNING]
> **ORACLE USES FUTURE DEVELOPMENT TRUTH / NOT DEPLOYABLE IN PRODUCTION**
>
> Phân tích dưới đây là một công cụ chẩn đoán ngoại tuyến thuần túy nhằm tìm ra **giới hạn trên lý thuyết (theoretical upper bound)** của khả năng cảnh báo sớm khi biết trước toàn bộ tương lai trong quá trình phát triển.
> - **Tuyệt đối cấm** sử dụng nhãn holdout hoặc dùng các quyết định cảnh báo của Oracle làm dự báo mô hình.
> - **Mục đích:** Đánh giá ảnh hưởng hình học của độ dài cửa sổ cảnh báo $H \in \{6\text{h}, 12\text{h}, 24\text{h}\}$ đối với bài toán bao phủ lỗi trong hạn mức ngân sách thực thể sản xuất $\le 30\%$.

In [5]:
# Formulation & Execution of MILP Oracle Diagnostic
def solve_oracle_diagnostic(rawspan, meta, budget_entities=263, time_limit_sec=20.0):
    """
    Formulate MILP to find hindsight coverage on development span.
    Variables:
      w_j in {0, 1} for each warning candidate interval j in 0..M-1
      c_i in {0, 1} for each unique raw observation i in 0..N-1
    Constraints:
      c_i <= sum_j (incidence_ji * w_j)  (covered only if at least one active interval covers it)
      sum_i (incidence_ji * c_i) >= n_j * w_j  (if interval j warned, all its covered entities are accounted)
      sum_i c_i <= budget_entities (workload budget constraint)
    Objective:
      Maximize sum_{i: fail_i=1} c_i
    """
    M = len(meta)
    N = len(rawspan)
    
    ts = rawspan["timestamp"].values
    starts = meta["target_start"].values[:, None]
    ends = meta["target_end"].values[:, None]
    inc = ((ts >= starts) & (ts < ends)).astype(int)
    
    n_interval = inc.sum(axis=1)
    fail_vec = rawspan["fail"].values
    
    # Objective vector: length M + N
    c_obj = np.zeros(M + N)
    c_obj[M:] = -fail_vec
    
    # Constraint 1: c_i - sum_j inc[j, i] * w_j <= 0  (N rows)
    A_1 = np.hstack([-inc.T, np.eye(N)])
    b_l_1 = np.full(N, -np.inf)
    b_u_1 = np.zeros(N)
    
    # Constraint 2: n_j * w_j - sum_i inc[j, i] * c_i <= 0 (M rows)
    A_2 = np.hstack([np.diag(n_interval), -inc])
    b_l_2 = np.full(M, -np.inf)
    b_u_2 = np.zeros(M)
    
    # Constraint 3: sum_i c_i <= budget_entities (1 row)
    A_3 = np.hstack([np.zeros((1, M)), np.ones((1, N))])
    b_l_3 = np.array([-np.inf])
    b_u_3 = np.array([budget_entities])
    
    A_tot = np.vstack([A_1, A_2, A_3])
    b_l_tot = np.concatenate([b_l_1, b_l_2, b_l_3])
    b_u_tot = np.concatenate([b_u_1, b_u_2, b_u_3])
    
    constraints = LinearConstraint(A_tot, b_l_tot, b_u_tot)
    integrality = np.ones(M + N)
    bounds = Bounds(lb=np.zeros(M + N), ub=np.ones(M + N))
    
    t0 = time.time()
    res = milp(c=c_obj, integrality=integrality, constraints=constraints, bounds=bounds,
               options={"time_limit": time_limit_sec})
    t_elapsed = time.time() - t0
    
    mip_gap = getattr(res, "mip_gap", np.nan)
    mip_dual_bound = getattr(res, "mip_dual_bound", np.nan)
    upper_fail_bound = -mip_dual_bound if np.isfinite(mip_dual_bound) else np.nan
    
    if res.x is not None:
        w_sol = (res.x[:M] > 0.5).astype(int)
        active_intervals = np.where(w_sol == 1)[0]
        if len(active_intervals) > 0:
            recomputed_cov = inc[active_intervals].any(axis=0)
            rec_workload = int(recomputed_cov.sum())
            rec_fail_cov = int((recomputed_cov * fail_vec).sum())
        else:
            rec_workload = 0
            rec_fail_cov = 0
            
        assert rec_workload <= budget_entities, f"Workload violation: {rec_workload} > {budget_entities}"
        assert abs(-res.fun - rec_fail_cov) < 1e-4, f"Objective mismatch: {-res.fun} vs {rec_fail_cov}"
        
        return {
            "status": res.status,
            "message": res.message,
            "time_sec": t_elapsed,
            "fail_covered": rec_fail_cov,
            "workload_entities": rec_workload,
            "n_intervals_active": int(w_sol.sum()),
            "is_optimal": (res.status == 0),
            "mip_gap": mip_gap,
            "mip_dual_bound": mip_dual_bound,
            "upper_fail_bound": upper_fail_bound
        }
    else:
        return {
            "status": res.status,
            "message": res.message,
            "time_sec": t_elapsed,
            "fail_covered": np.nan,
            "workload_entities": np.nan,
            "n_intervals_active": 0,
            "is_optimal": False,
            "mip_gap": np.nan,
            "mip_dual_bound": np.nan,
            "upper_fail_bound": np.nan
        }

# Run Oracle on fixed 3-span development benchmark [2008-08-21 18:00, 2008-10-02 18:00)
SPAN_START = pd.Timestamp("2008-08-21 18:00:00")
SPAN_END = pd.Timestamp("2008-10-02 18:00:00")

dev_span_df = df_raw[(df_raw["timestamp"] >= SPAN_START) & (df_raw["timestamp"] < SPAN_END)].copy().reset_index(drop=True)
total_dev_span_n = len(dev_span_df)
total_dev_span_fail = int(dev_span_df["fail"].sum())
budget_cap = int(np.floor(0.30 * total_dev_span_n))

print(f"Oracle Diagnostic Benchmark Setup:")
print(f"  Span: [{SPAN_START} to {SPAN_END})")
print(f"  Total Entities (N): {total_dev_span_n}, Total Fail: {total_dev_span_fail}")
print(f"  Max Workload Budget: {budget_cap} entities (<= 30.0%)\n")

oracle_summary = []
for H in [6, 12, 24]:
    # Allowed grid origins: target_start >= SPAN_START and target_end <= SPAN_END
    orig_start = SPAN_START - pd.Timedelta(hours=6)
    orig_end = SPAN_END - pd.Timedelta(hours=6 + H)
    origins_h = pd.date_range(start=orig_start, end=orig_end, freq="6h")
    
    meta_h = build_targets(dev_span_df, origins_h, horizon_h=H, lead_h=6, delay_h=6)
    
    res = solve_oracle_diagnostic(dev_span_df, meta_h, budget_entities=budget_cap, time_limit_sec=20.0)
    if np.isfinite(res["fail_covered"]):
        cov_pct = (res["fail_covered"] / total_dev_span_fail) * 100
        workload_pct = (res["workload_entities"] / total_dev_span_n) * 100
        fail_cov_str = f"{res['fail_covered']}/{total_dev_span_fail}"
        cov_str = f"{cov_pct:.1f}%"
        workload_str = f"{res['workload_entities']}/{total_dev_span_n}"
        workload_pct_str = f"{workload_pct:.1f}%"
    else:
        fail_cov_str = "No Solution"
        cov_str = "N/A"
        workload_str = "N/A"
        workload_pct_str = "N/A"
        
    upper_b_str = f"{res['upper_fail_bound']:.1f}" if np.isfinite(res["upper_fail_bound"]) else "N/A"
    gap_str = f"{res['mip_gap']:.4f}" if np.isfinite(res["mip_gap"]) else "N/A"
    
    oracle_summary.append({
        "Horizon H": f"{H}h",
        "Candidate Intervals": len(meta_h),
        "Active Intervals": res["n_intervals_active"],
        "Fail Covered": fail_cov_str,
        "Feasible Hindsight Coverage (%)": cov_str,
        "Workload Entities": workload_str,
        "Actual Workload (%)": workload_pct_str,
        "Upper Fail Bound": upper_b_str,
        "MIP Gap": gap_str,
        "Solver Status": res["message"],
        "Is Optimal": res["is_optimal"],
        "Time (s)": f"{res['time_sec']:.2f}"
    })

df_oracle_res = pd.DataFrame(oracle_summary)
print(df_oracle_res.to_string(index=False))


Oracle Diagnostic Benchmark Setup:
  Span: [2008-08-21 18:00:00 to 2008-10-02 18:00:00)
  Total Entities (N): 879, Total Fail: 35
  Max Workload Budget: 263 entities (<= 30.0%)



Horizon H  Candidate Intervals  Active Intervals Fail Covered Feasible Hindsight Coverage (%) Workload Entities Actual Workload (%) Upper Fail Bound MIP Gap                                                   Solver Status  Is Optimal Time (s)
       6h                  168                31        35/35                          100.0%           244/879               27.8%             35.0  0.0000 Optimization terminated successfully. (HiGHS Status 7: Optimal)        True     0.02
      12h                  167                22        34/35                           97.1%           257/879               29.2%             34.0  0.0000 Optimization terminated successfully. (HiGHS Status 7: Optimal)        True     0.16
      24h                  165                15        28/35                           80.0%           258/879               29.4%             28.0  0.0000 Optimization terminated successfully. (HiGHS Status 7: Optimal)        True     0.27
